# Goal 1 · Memory tools and turn order

**Goal:** register the missing memory tools on the governed MCP server, and correct the session read so the assistant can store and recall prior turns in chronological order.

**Files:** `src/memory_client.py`, `src/mcp_server.py`  |  **Evidence:** `mcp_tools_result.json`

### Concepts

- **The memory table.** `claims-sessions` has partition key `session_id` (S) and sort key `turn_id` (N). A `Query` on one `session_id` returns that session's items **sorted by the sort key**.
- **`ScanIndexForward`** chooses the direction: `True` is ascending (oldest turn first), `False` is descending (newest first).
- **MCP tools.** The assistant reaches its memory only through the governed MCP server (`FastMCP`). A function becomes a tool when it's decorated with `@mcp.tool()`. The tool name is the function name and the docstring is the description the model sees.

In [ ]:
import os, sys, uuid
sys.path.insert(0, os.path.abspath('.'))
import labkit
from src.memory_client import SessionMemoryStore

mem = SessionMemoryStore()
turns = mem.get_session('s-demo')
for t in turns:
    print(t['turn_id'], t['role'], '-', t['content'][:70])
print('order returned:', [t['turn_id'] for t in turns])

### Observe: it isn't only the display that's wrong

The broken state returns `[3, 2, 1]`. Now look at how `append_turn` picks the next id: it reads the session back and takes `existing[-1]["turn_id"] + 1`.

The next cell proves the consequence on a **scratch session**. **Never append test turns to `s-demo`**: the evidence expects exactly `[1, 2, 3]` there.

In [ ]:
labkit.show('src/memory_client.py', 'def append_turn', after=14)

In [ ]:
scratch = f's-scratch-{uuid.uuid4().hex[:8]}'
for i, (role, text) in enumerate([('user', 'first'), ('assistant', 'second'), ('user', 'third')], start=1):
    mem.seed_turn(scratch, i, role, text)                  # explicit ids 1, 2, 3

new_id = mem.append_turn(scratch, 'assistant', 'FOURTH (appended)')
print('append_turn assigned turn_id', new_id, '(it should be 4)')
for t in sorted(mem.get_session(scratch), key=lambda t: t['turn_id']):
    print(t['turn_id'], t['role'], '-', t['content'])


def drop_session(session_id):
    for t in mem.get_session(session_id):
        mem.table.delete_item(Key={'session_id': session_id, 'turn_id': t['turn_id']})

drop_session(scratch)
print(f'(scratch session {scratch} deleted)')

With descending order, `existing[-1]` is the **oldest** turn (id 1). The "next" id is therefore 2, and `PutItem` silently **overwrites turn 2**. A read-order bug becomes a memory-corruption bug.

## Task 1a · Read turns oldest first

In `src/memory_client.py`, fix `get_session` so it returns turns in ascending `turn_id` order, using DynamoDB itself rather than sorting in Python.

In [ ]:
labkit.show('src/memory_client.py', 'def get_session', after=18)

<details><summary>Hint 1</summary>Which argument of <code>table.query(...)</code> controls the sort direction of the sort key?</details>
<details><summary>Hint 2</summary><code>ScanIndexForward=True</code> means ascending (oldest first).</details>

Save the file, then reload and verify. The next cell re-imports the module, so your edit takes effect without restarting the kernel.

In [ ]:
memory_client = labkit.reload('memory_client')
mem = memory_client.SessionMemoryStore()
order = [t['turn_id'] for t in mem.get_session('s-demo')]
print('s-demo order:', order, '->', 'OK' if order == [1, 2, 3] else 'still wrong')

scratch = f's-scratch-{uuid.uuid4().hex[:8]}'
for i in (1, 2, 3):
    mem.seed_turn(scratch, i, 'user', f'turn {i}')
print('append_turn now assigns', mem.append_turn(scratch, 'assistant', 'turn 4'), '(expect 4)')
for t in mem.get_session(scratch):
    mem.table.delete_item(Key={'session_id': scratch, 'turn_id': t['turn_id']})

## Task 1b · Register the memory tools on the MCP server

In `src/mcp_server.py`, replace the `# TODO(DIY-1)` line with **two tools**, using the module's existing `memory` store:

| Tool name (exact) | Parameters | Returns |
|---|---|---|
| `append_turn` | `session_id: str, role: str, content: str` | a JSON string with `session_id` and the new `turn_id` |
| `get_session` | `session_id: str` | a JSON string: the list of turns, oldest first |

Follow the style of the other tools in the file: `@mcp.tool()`, type hints, a one-line docstring and `json.dumps(...)`.

In [ ]:
labkit.show('src/mcp_server.py', 'TODO(DIY-1)', before=10, after=3)

<details><summary>Hint 1</summary>Each tool is a thin wrapper: call <code>memory.append_turn(...)</code> or <code>memory.get_session(...)</code> and serialise the result with <code>json.dumps</code>.</details>
<details><summary>Hint 2</summary>FastMCP uses the <b>function name</b> as the tool name, so the functions must be called exactly <code>append_turn</code> and <code>get_session</code>.</details>

Importing `src.mcp_server` connects to all the stores (Neo4j, OpenSearch, PostgreSQL), just as the server does at start-up. Reload `memory_client` first, then `mcp_server`, which imports it.

In [ ]:
memory_client, mcp_server = labkit.reload('memory_client', 'mcp_server')
tools = sorted(t.name for t in await mcp_server.mcp.list_tools())
print('registered tools:', tools)
print('append_turn:', 'append_turn' in tools, '| get_session:', 'get_session' in tools)

Exercise the tools the way the assistant would, on a scratch session. `@mcp.tool()` returns the original function, so you can call it directly.

In [ ]:
scratch = f's-scratch-{uuid.uuid4().hex[:8]}'
labkit.pretty(mcp_server.append_turn(scratch, 'user', 'Is Nexus Tower covered for flood?'))
labkit.pretty(mcp_server.append_turn(scratch, 'assistant', 'Nexus Tower excludes catastrophic risk.'))
labkit.pretty(mcp_server.get_session(scratch))
for t in mcp_server.memory.get_session(scratch):
    mcp_server.memory.table.delete_item(Key={'session_id': scratch, 'turn_id': t['turn_id']})

## Publish the evidence

`emit_evidence mcp-tools` imports the MCP server, lists its tools and records the **unsorted** order `get_session('s-demo')` returns. It runs as a separate process, exactly as on the lab box.

In [ ]:
!{sys.executable} -m src.emit_evidence mcp-tools
labkit.upload('mcp_tools_result.json')
labkit.grade(1)

### If `s-demo` got extra or overwritten turns

Run the next cell. It deletes every `s-demo` turn and re-seeds the three original ones (the same `seed_turn` calls `main.seed()` makes). Then publish the evidence again.

In [ ]:
RESET_S_DEMO = False   # set True to run
if RESET_S_DEMO:
    m = memory_client.SessionMemoryStore()
    for t in m.get_session('s-demo'):
        m.table.delete_item(Key={'session_id': 's-demo', 'turn_id': t['turn_id']})
    m.seed_turn('s-demo', 1, 'user', 'I need to review the Nexus Tower commercial property policy.')
    m.seed_turn('s-demo', 2, 'assistant', 'Sure — Nexus Tower is a commercial property policy. What would you like to check?')
    m.seed_turn('s-demo', 3, 'user', 'Which governed metric applies to it?')
    print([t['turn_id'] for t in m.get_session('s-demo')])

### Debrief

- Ordering belongs to the **data store contract**. The class docstring promises ascending order. Rely on the sort key, not on a Python `sorted()` that hides the bug.
- Deriving ids by reading the session back is fragile under concurrency too. Two writers can compute the same `next_turn`. A production version would use a conditional write (`attribute_not_exists(turn_id)`) or an atomic counter.
- Tools are the assistant's only window onto memory. Name and describe them precisely, because the model chooses tools from those descriptions.